Look at predictions for $z=17$

In [2]:
from escripts import edata
from escripts import eMCMC
from escripts import eplots

In [3]:
eplots.evolving_UVLF_fit??

Signature:
eplots.evolving_UVLF_fit(
    my_UVLF,
    z_plot=None,
    plot_from_chain=True,
    nsamples=100,
    comparison_fits=[],
    comparison_labels=[],
    ncols=3,
    title='UVLF data vs. MCMC fit',
    burn_in=6000,
)
Source:   
def evolving_UVLF_fit(my_UVLF, z_plot = None, plot_from_chain = True, nsamples = 100, comparison_fits = [], comparison_labels = [], ncols = 3,
                           title = 'UVLF data vs. MCMC fit', burn_in = 6000):
    """
    Plot the UVLF at different redshifts
    Inputs:
        my_UVLF: eMCMC UVLF object
        z_plot [list of floats]: redshifts to plot, or leave as None if you want to plot all of them
        plot_from_chain [bool]: whether or not to plot the best fit and samples from the chain stored in my_UVLF. If False, only comparison_fits
                             parameter values will be plotted (so you can quickly check different fits this way).
        nsamples [int]: number of samples from the MCMC chain you want to appear i

In [ ]:
def extend_z(my_UVLF, z_plot = [17], fits = [], labels = [], ncols = 3, title = 'UVLF data vs. MCMC fit'):

    # Figure out how many subplots to make
    nz = len(z_plot)
    fig = plt.figure()
    gs = GridSpec(math.ceil((nz+1)/ncols), ncols, figure=fig)
    
    # Adjust size & spacing
    fig.set_size_inches(3*ncols, 1.25*(nz+1))
    plt.subplots_adjust(wspace = 0, hspace = 0.3)

    # Set the same y limits for all the plots
    ylo, yhi = np.log10(min([min(dat[3]) for dat in my_UVLF.data]))-0.25, np.log10(max([max(dat[3]) for dat in my_UVLF.data]))+0.25 

    # Get the x grid to evaluate the UVLF over
    plot_xdat, plot_xerr = my_UVLF.data[0][2], my_UVLF.data[0][5] # Use the lowest redshift MUV centers & bins as the grid to calculate the
                                                                # theoretical UVLF on

    i = 0 # Can't usse enumerate because some z-bins will be skipped
    for zbin in z_plot:
    
        zdat, zerr = zbin[0][0], zbin[0][1]

        ax = fig.add_subplot(gs[math.floor(i/ncols), i%ncols])
        
        # Plot comparison fits
        for fit, label, color, ls in zip(comparison_fits, comparison_labels, [eplots.turquoise, eplots.yellow, eplots.orange, eplots.green], 
                                        ['solid', 'dashdot', 'dashed', 'dotted']):
            ax.plot(plot_xdat, np.log10(my_UVLF.UVLF_wrapper(zdat,zerr,plot_xdat,plot_xerr, fit)), color = color, label = f'{label}', 
                    linestyle = ls, zorder = 1)

        # Plot data
        for dat, fmt in zip(zbin, ['o', 'v', 's']):
            _, _, xdat, ydat, yerr_upper, yerr_lower, xerr = edata.decompose(dat)
            yerr_lower, yerr_upper = estats.calc_log_error(ydat, yerr_lower), estats.calc_log_error(ydat, yerr_upper)
            ax.errorbar(xdat, np.log10(ydat), [yerr_lower, yerr_upper], fmt = fmt, capsize=0, markersize = 9, label = dat[7], 
                           markeredgecolor = 'none', markerfacecolor = navy, ecolor = navy)

        ax.invert_xaxis()
        ax.set_ylim(ylo, yhi)
        ax.set_xlim(np.max(plot_xdat)+0.25, np.min(plot_xdat)-0.25)
        ax.set_title(fr'$z \simeq {round(zdat, 1)}$', fontsize = 16, pad = 8)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True)) # Make it so that only integers can be used in the  
                                                                                       # axis labels
        if i%3 != 0:
            ax.axes.yaxis.set_ticklabels([])

        i += 1

    
    # Text & labels
    ylabel = r'$\log_{10}(\phi_{\rm{UV}}$ [$\rm{mag}^{-1} \rm{Mpc}^{-3}$])'
    xlabel = r'$M_{\rm{UV}}$ [mag]'

    if math.ceil((nz+1)/ncols) > 1:
        fig.supylabel(ylabel, x = 0.05)
        fig.supxlabel(xlabel, y = 0.07)
        fig.text(.5, 0.915, title, ha = 'center', fontsize = 20)
    else:
        if nz == 1:
            ax.set_ylabel(ylabel)
            ax.set_xlabel(xlabel)
            ax.set_title(title)
        else:
            fig.text(0.065, 0.225, ylabel, rotation = 'vertical')
            fig.text(0.375, 0, xlabel, ha = 'center')
            fig.text(0.375, 1, title, ha = 'center', fontsize = 20)

    # Put legend outside the last axis
    fig_ax = fig.add_subplot(gs[math.floor(i/ncols), i%ncols])
    fig_ax.axis("off")  
    handles, labels = ax.get_legend_handles_labels() # Grab handles/labels from the real axes
    fig_ax.legend(handles, labels, loc='center')
    
    return fig